# Tutorial 12: Unified Embedding with embed_adata
The `embed_adata()` method is the **single entry point** for computing
both **cell-level** and **perturbation-level** embeddings on an AnnData
object in one call.

- **Cell embeddings** come from expression data (PCA, scVI, scGPT, Geneformer, ...)
- **Perturbation embeddings** come from perturbation annotations in `.obs`
  (gene symbols -> DNA/protein embeddings, SMILES -> molecule embeddings)

All results are stored in `.obsm` with a consistent `X_` prefix.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
import anndata as ad
import numpy as np
import scipy.sparse as sp
import pandas as pd

from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Load a Perturb-seq Dataset

We use a small subset of a Perturb-seq dataset. The key column is
the perturbation annotation in `.obs` (e.g. gene symbols or SMILES).

In [ ]:
# Load example dataset (adjust path as needed)
# adata = ad.read_h5ad("path/to/perturbseq.h5ad")

# For demonstration, create a synthetic dataset
rng = np.random.default_rng(42)
n_cells, n_genes = 500, 2000
X = sp.random(n_cells, n_genes, density=0.1, format="csr",
              random_state=42, dtype=np.float32)
X.data = np.abs(X.data) * 100  # make count-like

gene_names = [f"Gene_{i}" for i in range(n_genes)]
perturbations = rng.choice(
    ["TP53", "BRCA1", "EGFR", "MYC", "control"],
    size=n_cells,
)

adata = ad.AnnData(
    X=X,
    obs=pd.DataFrame({"perturbation": perturbations}),
    var=pd.DataFrame(index=gene_names),
)
adata.obs.index = [f"cell_{i}" for i in range(n_cells)]

print(adata)
print(f"Perturbations: {adata.obs['perturbation'].value_counts().to_dict()}")

## 2. Cell Embeddings Only

When you only need expression-based embeddings, pass `cell_models`
without `perturbation_models`.

In [ ]:
result = embedder.embed_adata(
    adata,
    cell_models=["pca"],
    preprocessing="standard",
    n_pca_components=30,
    n_top_genes=1000,
)

print(f".X (raw counts): {result.X.shape}")
print(f"Layers: {list(result.layers.keys())}")
print(f".obsm keys: {list(result.obsm.keys())}")
print(f"PCA shape: {result.obsm['X_pca'].shape}")

## 3. Perturbation Embeddings Only

When you want to embed the perturbation identifiers (gene symbols,
SMILES, etc.) without computing cell embeddings from expression,
pass `perturbation_models` and set `preprocessing="none"`.

In [ ]:
result_pert = embedder.embed_adata(
    adata,
    perturbation_models=["esm2_650M"],
    perturbation_column="perturbation",
    perturbation_type="symbol",
    preprocessing="none",
)

print(f".obsm keys: {list(result_pert.obsm.keys())}")
print(f"ESM2 shape: {result_pert.obsm['X_esm2_650M'].shape}")

# Check metadata
meta = result_pert.uns["embpy_embeddings"]["esm2_650M"]
print(f"Perturbations embedded: {meta['n_perturbations_embedded']}/{meta['n_perturbations_total']}")
print(f"Cells mapped: {meta['n_cells_mapped']}/{meta['n_cells']}")

## 4. Combined Cell + Perturbation Embeddings

The real power of `embed_adata()` is computing everything in one call.
Cell embeddings and perturbation embeddings are stored side by side
in `.obsm`.

In [ ]:
result_combined = embedder.embed_adata(
    adata,
    # Cell-level
    cell_models=["pca"],
    preprocessing="standard",
    n_pca_components=30,
    n_top_genes=1000,
    # Perturbation-level
    perturbation_models=["esm2_650M"],
    perturbation_column="perturbation",
    perturbation_type="symbol",
)

print("All .obsm keys:")
for key in sorted(result_combined.obsm.keys()):
    print(f"  {key}: {result_combined.obsm[key].shape}")

print(f"\n.X (raw counts): {result_combined.X.shape}")
print(f"Layers: {list(result_combined.layers.keys())}")
print(f"\nMetadata keys: {list(result_combined.uns['embpy_embeddings'].keys())}")

## 5. Auto-Detecting Perturbation Types

Set `perturbation_type="auto"` to let embpy auto-detect whether each
perturbation identifier is a gene symbol, Ensembl ID, or SMILES string.
This is useful for datasets with mixed perturbation types.

In [ ]:
from embpy.resources.gene import detect_identifier_type

examples = ["TP53", "ENSG00000141510", "CCO", "BRCA1"]
for ident in examples:
    print(f"  {ident:25s} -> {detect_identifier_type(ident)}")

## 6. Inspecting Embedding Metadata

After `embed_adata()`, all metadata is stored in
`adata.uns["embpy_embeddings"]`. This includes embedding dimensions,
success counts, and the wrapper class used.

In [ ]:
import json

if "embpy_embeddings" in result_combined.uns:
    for model_key, info in result_combined.uns["embpy_embeddings"].items():
        print(f"\n{model_key}:")
        for k, v in info.items():
            print(f"  {k}: {v}")

## 7. Stacking independently built perturbation spaces

`embed_adata()` is the right tool when your perturbation identifiers all
live in the same `.obs` column of one AnnData. But sometimes you have two
*separate* perturbation experiments (e.g. a CRISPR gene-KO screen and a
small-molecule screen) already embedded with different models. You can
still place them in a shared analysis space with
`PerturbationProcessor.combine_perturbation_spaces`:

1. PCA-reduce each embedding matrix to a **common dimensionality**.
2. Expose the reduced matrix under a **shared `.obsm` key**.
3. Stack the two AnnDatas; a `perturbation_type` column is added to
   `.obs` so each row is labelled with its source.

The result is a single AnnData you can cluster, visualise, or pass to
any downstream tool that expects one `obsm` key across all rows.

In [ ]:
from embpy.pp.basic import PerturbationProcessor, reduce_embeddings
import embpy.pl as epl

pp = PerturbationProcessor(embedder=embedder)

# (a) Gene-perturbation AnnData: embed 8 cancer genes with ESM-2.
gene_list = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC", "PTEN", "RB1", "AKT1"]
adata_genes = pp.build_embedding_matrix(
    identifiers=gene_list,
    model="esm2_650M",
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
    obsm_key="X_esm2",
)
adata_genes = pp.filter_failed_embeddings(adata_genes, obsm_key="X_esm2")

# (b) Drug-perturbation AnnData: embed 5 drug names with ChemBERTa.
drug_names = ["aspirin", "ibuprofen", "caffeine", "paclitaxel", "metformin"]
adata_drugs = pp.build_molecule_embedding_matrix(
    identifiers=drug_names,
    id_type="drug_name",
    model="chemberta2MTR",
    pooling_strategy="mean",
    obsm_key="X_chemberta",
)
adata_drugs = pp.filter_failed_embeddings(adata_drugs, obsm_key="X_chemberta")

# (c) PCA-reduce both to the SAME dimensionality, expose under a shared key.
# ad.concat (used internally by combine_perturbation_spaces) requires
# matching obsm widths, and each PCA is capped at n_obs - 1 components.
N_DIMS = min(adata_genes.n_obs, adata_drugs.n_obs) - 1
adata_genes = reduce_embeddings(adata_genes, obsm_key="X_esm2",
                                 n_components=N_DIMS, scale=True)
adata_drugs = reduce_embeddings(adata_drugs, obsm_key="X_chemberta",
                                 n_components=N_DIMS, scale=True)
adata_genes.obsm["X_combined"] = adata_genes.obsm["X_esm2_pca"]
adata_drugs.obsm["X_combined"] = adata_drugs.obsm["X_chemberta_pca"]

# (d) Stack. A `perturbation_type` column tells you which row came from where.
combined = pp.combine_perturbation_spaces(
    adata_genes, adata_drugs,
    labels=["gene", "drug"],
    obsm_key="X_combined",
)

print(combined)
print()
print("perturbation_type counts:")
print(combined.obs["perturbation_type"].value_counts())

In [ ]:
import embpy.tl as etl

# 13 rows total is too few for UMAP's default n_neighbors=15, so we precompute
# the layout with a smaller neighbourhood and then hand the cached basis to
# `plot_embedding_space`. Same pattern for any small embedding matrix.
etl.compute_umap(
    combined,
    obsm_key="X_combined",
    n_neighbors=5,
    output_key="X_umap_combined",
)

fig = epl.plot_embedding_space(
    combined,
    basis="X_umap_combined",
    color="perturbation_type",
    title="Combined gene + drug perturbation space (UMAP)",
    figsize=(7, 5),
)
fig.show()

## Summary

| What | How |
|---|---|
| Cell embeddings only | `embed_adata(adata, cell_models=["pca", "scvi"])` |
| Perturbation embeddings only | `embed_adata(adata, perturbation_models=["esm2_650M"], perturbation_column="gene")` |
| Both in one call | `embed_adata(adata, cell_models=[...], perturbation_models=[...], perturbation_column="...")` |
| Auto-detect perturbation type | `perturbation_type="auto"` |
| Results location | `.obsm["X_{model_name}"]` |
| Metadata | `.uns["embpy_embeddings"]` |
| Stack two pre-embedded perturbation sets | `PerturbationProcessor.combine_perturbation_spaces(a, b, labels=[...], obsm_key=...)` |